In [1]:
# Add to FEATURE_COLS after feature_bmi:
# 'feature_waist_cm'

# In feature engineering:
df['feature_waist_cm'] = df['waist_cm'].clip(50, 160)

# Expected sign: positive (higher waist = higher risk)
# Particularly important for South Asian risk calibration


NameError: name 'df' is not defined

In [3]:

import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('nhanes_data')
CYCLES_LABELS = {'2011-12':'G', '2013-14':'H', '2015-16':'I', '2017-18':'J'}

# Reload feature dataset
df = pd.read_csv('nhanes_data/nhanes_features.csv')
cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')

# Add age and BMI
df['feature_age'] = df['age_years'].clip(20, 80)
df['feature_bmi'] = df['bmi'].clip(15, 60)

# Add waist circumference from cleaned
df = df.drop(columns=[c for c in df.columns if c in
             ['waist_cm','gender','hdl','triglycerides']], errors='ignore')
df = df.merge(cleaned[['participant_id','waist_cm','gender',
                        'hdl','triglycerides']], on='participant_id', how='left')

# Add sedentary hours from PAQ
paq_frames = []
for cycle, suffix in CYCLES_LABELS.items():
    f = DATA_DIR / f'PAQ_{suffix}.XPT'
    if f.exists():
        df_paq = pd.read_sas(str(f), format='xport', encoding='utf-8')
        keep = {'SEQN':'participant_id','PAD680':'sedentary_mins_per_day',
                'PAQ605':'vigorous_activity','PAQ620':'moderate_activity'}
        df_paq = df_paq[[c for c in keep if c in df_paq.columns]].rename(columns=keep)
        paq_frames.append(df_paq)
paq = pd.concat(paq_frames, ignore_index=True)
df = df.merge(paq[['participant_id','sedentary_mins_per_day']],
              on='participant_id', how='left')
df['feature_sedentary_hrs'] = (
    df['sedentary_mins_per_day'].clip(0,960)/60).fillna(8.0)

# Add waist circumference feature
df['feature_waist_cm'] = df['waist_cm'].clip(50, 160)

print('Dataset ready:', df.shape)
print('Waist cm available:', df['feature_waist_cm'].notna().sum())
print('Waist range:', df['feature_waist_cm'].min(), 'to', df['feature_waist_cm'].max())
print('\nCorrelation of waist_cm with diabetes outcome:')
print(df[['feature_waist_cm','feature_bmi','outcome_diabetes']].corr()['outcome_diabetes'].round(3))


Dataset ready: (18835, 24)
Waist cm available: 18288
Waist range: 55.5 to 160.0

Correlation of waist_cm with diabetes outcome:
feature_waist_cm    0.286
feature_bmi         0.234
outcome_diabetes    1.000
Name: outcome_diabetes, dtype: float64


In [4]:
FEATURE_COLS_V4 = [
    'feature_age',
    'feature_bmi',
    'feature_waist_cm',
    'feature_sedentary_hrs',
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

model_df = df[FEATURE_COLS_V4 + ['outcome_diabetes','race_ethnicity']].dropna()
print(f'Dataset with waist: {model_df.shape}')

X = model_df[FEATURE_COLS_V4].values
y = model_df['outcome_diabetes'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

gb_v4 = GradientBoostingClassifier(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42
)
print('Training... (2 minutes)')
gb_v4.fit(X_tr, y_tr)

auc = roc_auc_score(y_te, gb_v4.predict_proba(X_te)[:,1])
print(f'Diabetes AUC with waist: {auc:.3f} (previous: 0.765)')
print(f'Change: {auc - 0.765:+.3f}')

asian = model_df[model_df['race_ethnicity']==6]
asian_auc = roc_auc_score(asian['outcome_diabetes'],
              gb_v4.predict_proba(asian[FEATURE_COLS_V4].values)[:,1])
print(f'Asian AUC with waist: {asian_auc:.3f} (previous: 0.864)')
print(f'Change: {asian_auc - 0.864:+.3f}')

importance_df = pd.DataFrame({
    'feature': FEATURE_COLS_V4,
    'importance': gb_v4.feature_importances_
}).sort_values('importance', ascending=False)
print('\nFeature importances:')
print(importance_df.to_string(index=False))

Dataset with waist: (18288, 13)
Training... (2 minutes)
Diabetes AUC with waist: 0.780 (previous: 0.765)
Change: +0.015
Asian AUC with waist: 0.871 (previous: 0.864)
Change: +0.007

Feature importances:
                   feature  importance
               feature_age    0.472187
          feature_waist_cm    0.136536
               feature_bmi    0.081690
    feature_mufa_sfa_ratio    0.043460
feature_refined_carb_share    0.043228
feature_fiber_per_1000kcal    0.042271
    feature_sfa_pct_energy    0.041697
feature_protein_pct_energy    0.041641
     feature_glycemic_load    0.040419
         feature_sodium_mg    0.038783
     feature_sedentary_hrs    0.018088


In [5]:
import json, numpy as np

# Calibration
np.random.seed(42)
n = 5000
X_rand = np.column_stack([
    np.random.uniform(25, 70, n),      # age
    np.random.uniform(17, 38, n),      # bmi
    np.random.uniform(60, 120, n),     # waist_cm
    np.random.uniform(3, 12, n),       # sedentary
    np.random.uniform(30, 350, n),     # gl
    np.random.uniform(0.05, 0.99, n),  # refined carb
    np.random.uniform(1, 30, n),       # fiber
    np.random.uniform(0.05, 0.35, n),  # protein
    np.random.uniform(0.02, 0.25, n),  # sfa
    np.random.uniform(0.1, 5.0, n),    # mufa:sfa
    np.random.uniform(500, 6000, n),   # sodium
])
probs = gb_v4.predict_proba(X_rand)[:,1]
p5, p95 = np.percentile(probs, 5), np.percentile(probs, 95)
print(f'Calibration: p5={p5:.3f}, p95={p95:.3f}')

def cal_score(model, feat_dict, cols, p5, p95):
    x = np.array([feat_dict.get(c,0) for c in cols]).reshape(1,-1)
    prob = model.predict_proba(x)[0][1]
    return int(np.clip(round(10 + (prob-p5)/(p95-p5)*80), 0, 100))

# Test profiles
young_healthy = {'feature_age':30,'feature_bmi':21,'feature_waist_cm':75,
                 'feature_sedentary_hrs':4,'feature_glycemic_load':90,
                 'feature_refined_carb_share':0.05,'feature_fiber_per_1000kcal':18,
                 'feature_protein_pct_energy':0.18,'feature_sfa_pct_energy':0.04,
                 'feature_mufa_sfa_ratio':2.8,'feature_sodium_mg':1200}
middle_good = {'feature_age':50,'feature_bmi':26,'feature_waist_cm':90,
               'feature_sedentary_hrs':7,'feature_glycemic_load':120,
               'feature_refined_carb_share':0.30,'feature_fiber_per_1000kcal':12,
               'feature_protein_pct_energy':0.15,'feature_sfa_pct_energy':0.07,
               'feature_mufa_sfa_ratio':1.5,'feature_sodium_mg':2000}
middle_poor = {'feature_age':50,'feature_bmi':26,'feature_waist_cm':90,
               'feature_sedentary_hrs':7,'feature_glycemic_load':280,
               'feature_refined_carb_share':0.90,'feature_fiber_per_1000kcal':3,
               'feature_protein_pct_energy':0.08,'feature_sfa_pct_energy':0.14,
               'feature_mufa_sfa_ratio':0.4,'feature_sodium_mg':3500}
older_high = {'feature_age':65,'feature_bmi':30,'feature_waist_cm':105,
              'feature_sedentary_hrs':10,'feature_glycemic_load':250,
              'feature_refined_carb_share':0.85,'feature_fiber_per_1000kcal':4,
              'feature_protein_pct_energy':0.09,'feature_sfa_pct_energy':0.13,
              'feature_mufa_sfa_ratio':0.5,'feature_sodium_mg':3000}

print('\nCalibrated scores:')
print(f'Young healthy (waist 75cm):       {cal_score(gb_v4, young_healthy, FEATURE_COLS_V4, p5, p95)}')
print(f'Middle, good diet (waist 90cm):   {cal_score(gb_v4, middle_good, FEATURE_COLS_V4, p5, p95)}')
print(f'Middle, poor diet (waist 90cm):   {cal_score(gb_v4, middle_poor, FEATURE_COLS_V4, p5, p95)}')
print(f'Older, high risk (waist 105cm):   {cal_score(gb_v4, older_high, FEATURE_COLS_V4, p5, p95)}')

# Save
with open('nhanes_data/diabetes_final.pkl', 'wb') as f:
    pickle.dump(gb_v4, f)

meta = {
    'model_type': 'gradient_boosting',
    'purpose': 'personal_risk_context',
    'feature_cols': FEATURE_COLS_V4,
    'test_auc': float(auc),
    'asian_auc': float(asian_auc),
    'calibration': {'p5': float(p5), 'p95': float(p95)},
    'improvement_note': 'V4 adds waist_cm — better visceral fat proxy for South Asians. AUC 0.780, Asian AUC 0.871.',
    'ui_description': (
        'Based on your age, waist circumference, BMI, activity level, and dietary '
        'patterns compared to a population with known health outcomes (NHANES 2011-2018).'
    ),
}
with open('nhanes_data/diabetes_final_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\nSaved diabetes_final.pkl (AUC {auc:.3f}, Asian AUC {asian_auc:.3f})')
print('Previous: AUC 0.765, Asian AUC 0.864')
print(f'Improvement: +{auc-0.765:.3f} overall, +{asian_auc-0.864:.3f} Asian')

Calibration: p5=0.037, p95=0.656

Calibrated scores:
Young healthy (waist 75cm):       10
Middle, good diet (waist 90cm):   26
Middle, poor diet (waist 90cm):   37
Older, high risk (waist 105cm):   76

Saved diabetes_final.pkl (AUC 0.780, Asian AUC 0.871)
Previous: AUC 0.765, Asian AUC 0.864
Improvement: +0.015 overall, +0.007 Asian


In [6]:
# CVD model with waist circumference
df['low_hdl'] = (
    ((df['gender'] == 1) & (df['hdl'] < 40)) |
    ((df['gender'] == 2) & (df['hdl'] < 50))
).astype(int)
df['high_trig'] = (df['triglycerides'] > 150).astype(int)
df['outcome_cvd_v2'] = ((df['high_trig'] == 1) | (df['low_hdl'] == 1)).astype(int)

model_df_cvd = df[FEATURE_COLS_V4 + ['outcome_cvd_v2','race_ethnicity']].dropna()
print(f'CVD dataset: {model_df_cvd.shape}')
print(f'CVD prevalence: {model_df_cvd["outcome_cvd_v2"].mean():.3f}')

X_cvd = model_df_cvd[FEATURE_COLS_V4].values
y_cvd = model_df_cvd['outcome_cvd_v2'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cvd, y_cvd, test_size=0.2, random_state=42, stratify=y_cvd)

gb_cvd_v4 = GradientBoostingClassifier(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42
)
print('Training CVD model...')
gb_cvd_v4.fit(X_tr, y_tr)

cvd_auc = roc_auc_score(y_te, gb_cvd_v4.predict_proba(X_te)[:,1])
print(f'CVD AUC with waist: {cvd_auc:.3f} (previous: 0.673)')
print(f'Change: {cvd_auc - 0.673:+.3f}')

asian_cvd = model_df_cvd[model_df_cvd['race_ethnicity']==6]
cvd_asian_auc = roc_auc_score(asian_cvd['outcome_cvd_v2'],
                  gb_cvd_v4.predict_proba(asian_cvd[FEATURE_COLS_V4].values)[:,1])
print(f'CVD Asian AUC with waist: {cvd_asian_auc:.3f} (previous: 0.781)')
print(f'Change: {cvd_asian_auc - 0.781:+.3f}')

importance_df = pd.DataFrame({
    'feature': FEATURE_COLS_V4,
    'importance': gb_cvd_v4.feature_importances_
}).sort_values('importance', ascending=False)
print('\nCVD Feature importances:')
print(importance_df.to_string(index=False))

# Calibration
X_rand_cvd = np.column_stack([
    np.random.uniform(25, 70, n),
    np.random.uniform(17, 38, n),
    np.random.uniform(60, 120, n),
    np.random.uniform(3, 12, n),
    np.random.uniform(30, 350, n),
    np.random.uniform(0.05, 0.99, n),
    np.random.uniform(1, 30, n),
    np.random.uniform(0.05, 0.35, n),
    np.random.uniform(0.02, 0.25, n),
    np.random.uniform(0.1, 5.0, n),
    np.random.uniform(500, 6000, n),
])
probs_cvd = gb_cvd_v4.predict_proba(X_rand_cvd)[:,1]
p5_cvd  = np.percentile(probs_cvd, 5)
p95_cvd = np.percentile(probs_cvd, 95)
print(f'\nCVD calibration: p5={p5_cvd:.3f}, p95={p95_cvd:.3f}')

print('\nCVD calibrated scores:')
print(f'Young healthy:      {cal_score(gb_cvd_v4, young_healthy, FEATURE_COLS_V4, p5_cvd, p95_cvd)}')
print(f'Middle, good diet:  {cal_score(gb_cvd_v4, middle_good, FEATURE_COLS_V4, p5_cvd, p95_cvd)}')
print(f'Middle, poor diet:  {cal_score(gb_cvd_v4, middle_poor, FEATURE_COLS_V4, p5_cvd, p95_cvd)}')
print(f'Older, high risk:   {cal_score(gb_cvd_v4, older_high, FEATURE_COLS_V4, p5_cvd, p95_cvd)}')

with open('nhanes_data/cvd_final.pkl', 'wb') as f:
    pickle.dump(gb_cvd_v4, f)

cvd_meta = {
    'model_type': 'gradient_boosting',
    'purpose': 'personal_risk_context',
    'feature_cols': FEATURE_COLS_V4,
    'test_auc': float(cvd_auc),
    'asian_auc': float(cvd_asian_auc),
    'calibration': {'p5': float(p5_cvd), 'p95': float(p95_cvd)},
    'outcome_definition': 'Triglycerides > 150 OR low HDL (men <40, women <50)',
    'improvement_note': 'V4 adds waist_cm. AUC {:.3f}, Asian AUC {:.3f}'.format(cvd_auc, cvd_asian_auc),
    'ui_description': (
        'Based on your age, waist circumference, BMI, activity level, and dietary '
        'patterns compared to a population with known cardiovascular risk markers (NHANES 2011-2018).'
    ),
}
with open('nhanes_data/cvd_final_meta.json', 'w') as f:
    json.dump(cvd_meta, f, indent=2)

print(f'\nSaved cvd_final.pkl (AUC {cvd_auc:.3f}, Asian AUC {cvd_asian_auc:.3f})')

CVD dataset: (18288, 13)
CVD prevalence: 0.339
Training CVD model...
CVD AUC with waist: 0.674 (previous: 0.673)
Change: +0.001
CVD Asian AUC with waist: 0.781 (previous: 0.781)
Change: +0.000

CVD Feature importances:
                   feature  importance
               feature_bmi    0.208225
          feature_waist_cm    0.177935
feature_refined_carb_share    0.104656
feature_fiber_per_1000kcal    0.098125
    feature_mufa_sfa_ratio    0.069441
               feature_age    0.068082
     feature_glycemic_load    0.063842
feature_protein_pct_energy    0.062783
         feature_sodium_mg    0.062339
    feature_sfa_pct_energy    0.057554
     feature_sedentary_hrs    0.027019

CVD calibration: p5=0.053, p95=0.698

CVD calibrated scores:
Young healthy:      11
Middle, good diet:  18
Middle, poor diet:  19
Older, high risk:   28

Saved cvd_final.pkl (AUC 0.674, Asian AUC 0.781)


In [7]:
import json, pickle, numpy as np
from pathlib import Path

with open('nhanes_data/diabetes_final.pkl', 'rb') as f:
    gb_d = pickle.load(f)
with open('nhanes_data/cvd_final.pkl', 'rb') as f:
    gb_c = pickle.load(f)
with open('nhanes_data/diabetes_final_meta.json') as f:
    dm = json.load(f)
with open('nhanes_data/cvd_final_meta.json') as f:
    cm = json.load(f)

FEATURE_COLS = dm['feature_cols']
print('Features:', FEATURE_COLS)
print('Diabetes AUC:', dm['test_auc'], '| Asian AUC:', dm['asian_auc'])
print('CVD AUC:', cm['test_auc'], '| Asian AUC:', cm['asian_auc'])

def export_tree(tree):
    t = tree.tree_
    def recurse(node):
        if t.children_left[node] == -1:
            return {'leaf': float(t.value[node][0][0])}
        return {
            'feature': int(t.feature[node]),
            'threshold': float(t.threshold[node]),
            'left': recurse(t.children_left[node]),
            'right': recurse(t.children_right[node]),
        }
    return recurse(0)

def export_gbm_model(model, meta, label):
    print(f'Exporting {label}...')
    trees = [[export_tree(tree) for tree in stage]
             for stage in model.estimators_]
    return {
        'label': label,
        'feature_cols': meta['feature_cols'],
        'n_estimators': int(model.n_estimators),
        'learning_rate': float(model.learning_rate),
        'init_score': float(model.init_.class_prior_[1]),
        'calibration': meta['calibration'],
        'test_auc': meta['test_auc'],
        'asian_auc': meta.get('asian_auc'),
        'ui_description': meta.get('ui_description', ''),
        'trees': trees,
    }

diabetes_export = export_gbm_model(gb_d, dm, 'diabetes')
cvd_export      = export_gbm_model(gb_c, cm, 'cvd')

ml_weights = {
    'version': '3.0',
    'model_type': 'gradient_boosting',
    'purpose': 'personal_risk_context',
    'trained_on': 'NHANES 2011-2018',
    'feature_cols': FEATURE_COLS,
    'changelog': 'V3 adds waist_cm — better visceral fat proxy for South Asians. Diabetes AUC 0.780 (+0.015), Asian AUC 0.871 (+0.007).',
    'diabetes': diabetes_export,
    'cvd': cvd_export,
}

out_path = Path('../data/ml_weights.json')
with open(out_path, 'w') as f:
    json.dump(ml_weights, f)

size_kb = out_path.stat().st_size / 1024
print(f'\nSaved: {out_path.resolve()}')
print(f'File size: {size_kb:.0f} KB')

# Verify round-trip
def js_inference(weights_block, features_dict):
    feature_cols = weights_block['feature_cols']
    x = [features_dict.get(c, 0) for c in feature_cols]
    score = weights_block['init_score']
    lr = weights_block['learning_rate']
    for stage in weights_block['trees']:
        for tree in stage:
            node = tree
            while 'leaf' not in node:
                if x[node['feature']] <= node['threshold']:
                    node = node['left']
                else:
                    node = node['right']
            score += lr * node['leaf']
    prob = 1 / (1 + np.exp(-score))
    p5  = weights_block['calibration']['p5']
    p95 = weights_block['calibration']['p95']
    return int(np.clip(round(10 + (prob-p5)/(p95-p5)*80), 0, 100))

test_high = {
    'feature_age': 55, 'feature_bmi': 30, 'feature_waist_cm': 105,
    'feature_sedentary_hrs': 10, 'feature_glycemic_load': 280,
    'feature_refined_carb_share': 0.90, 'feature_fiber_per_1000kcal': 3,
    'feature_protein_pct_energy': 0.08, 'feature_sfa_pct_energy': 0.14,
    'feature_mufa_sfa_ratio': 0.4, 'feature_sodium_mg': 3500,
}
test_low = {
    'feature_age': 30, 'feature_bmi': 21, 'feature_waist_cm': 75,
    'feature_sedentary_hrs': 4, 'feature_glycemic_load': 90,
    'feature_refined_carb_share': 0.05, 'feature_fiber_per_1000kcal': 18,
    'feature_protein_pct_energy': 0.18, 'feature_sfa_pct_energy': 0.04,
    'feature_mufa_sfa_ratio': 2.8, 'feature_sodium_mg': 1200,
}

d_high = js_inference(ml_weights['diabetes'], test_high)
d_low  = js_inference(ml_weights['diabetes'], test_low)
c_high = js_inference(ml_weights['cvd'], test_high)
c_low  = js_inference(ml_weights['cvd'], test_low)

print(f'\nVerification:')
print(f'High-risk — Diabetes: {d_high}  CVD: {c_high}')
print(f'Low-risk  — Diabetes: {d_low}   CVD: {c_low}')
print(f'Directionality correct: {d_high > d_low and c_high > c_low}')


Features: ['feature_age', 'feature_bmi', 'feature_waist_cm', 'feature_sedentary_hrs', 'feature_glycemic_load', 'feature_refined_carb_share', 'feature_fiber_per_1000kcal', 'feature_protein_pct_energy', 'feature_sfa_pct_energy', 'feature_mufa_sfa_ratio', 'feature_sodium_mg']
Diabetes AUC: 0.7803004747626185 | Asian AUC: 0.8710691997265531
CVD AUC: 0.6744437484018154 | Asian AUC: 0.7810474625477516
Exporting diabetes...
Exporting cvd...

Saved: C:\Users\obbha\OneDrive\South-asian-diet-risk.cd\data\ml_weights.json
File size: 821 KB

Verification:
High-risk — Diabetes: 86  CVD: 43
Low-risk  — Diabetes: 16   CVD: 23
Directionality correct: True
